In [ ]:
!pip install -qU langchain langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.5 MB/s eta 0:00:00


In [ ]:
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.4 MB/s eta 0:00:00


In [ ]:
import os, sys, platform
from google.colab import userdata
from datetime import datetime

In [ ]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
import langchain, langchain_core
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Partie 1 :

In [ ]:
model_name = "llama-3.3-70b-versatile"

llm = ChatGroq(
    model=model_name,
    temperature=0.99
)
print(f" Ready with model: {model_name}")

 Ready with model: llama-3.3-70b-versatile


In [ ]:
# First LLM call.
msg = "Explain AI in one friendly paragraph for a complete beginner."
resp = llm.invoke(msg)
print(resp.content)

Here's a friendly introduction to AI: Artificial Intelligence, or AI for short, refers to computers or machines that can think and learn like humans. Imagine you have a super-smart robot that can help you with tasks, like recognizing pictures, understanding what you say, or even playing games with you. AI is like a brain for these robots, allowing them to get better and better at doing things on their own, without needing to be told exactly what to do every time. It's made possible by complex algorithms and lots of data, which the AI uses to learn and improve. The goal of AI is to make life easier and more fun for us, by helping with everything from simple tasks like responding to emails, to more complex things like driving cars or helping doctors diagnose diseases. And the best part is, AI is always improving, so it's an exciting time to explore what it can do and how it can help us!


## Simple Chain

In [ ]:
# Build a motivational quote generator with a parameterized prompt
motivation_prompt = PromptTemplate.from_template(
    "Write a short, uplifting motivational quote for someone working as a {profession}. "
    "Keep it under 25 words."
)

In [ ]:
chain = motivation_prompt | llm | StrOutputParser()

In [ ]:
print("Exemple : ")
print(chain.invoke({"profession": "coder"}))

Exemple : 
"Code with passion, innovate with purpose, and create with joy."


## SequentialChain

In [ ]:
# Step 1: generate ideas
idea_prompt = PromptTemplate.from_template(
    "Generate 3 creative ideas about: {topic}. Return them as a numbered list."
)

# Step 2: summarize the ideas
summary_prompt = PromptTemplate.from_template(
    "Summarize the ideas below in 2 friendly sentences for a beginner:\n\n{ideas}"
)

In [ ]:
sequential_chain = idea_prompt | llm | summary_prompt | llm | StrOutputParser()

In [ ]:
print(sequential_chain.invoke({"topic": "coding"}))

Here are some exciting ideas about coding that you might enjoy: you can use coding to create amazing interactive art exhibits that respond to your movements and emotions, or develop platforms that help solve real-world social and environmental issues. You can even learn to code by playing fun games that teach you programming concepts through puzzles and virtual world-building, making it a fun and engaging experience to develop your coding skills.


## Correction :

In [ ]:
idea_chain = idea_prompt | llm | StrOutputParser()

In [ ]:
# Compose a chain where the output of idea_chain is fed into the summary prompt
sequential_chain = {
    "ideas": idea_chain,
    "topic": RunnablePassthrough()
} | summary_prompt | llm | StrOutputParser()

In [ ]:
print("corrigé : ")
print(sequential_chain.invoke("using AI to improve school homework feedback"))

corrigé : 
Here are 2 friendly sentences summarizing the ideas for a beginner: 
Using AI to improve school homework feedback is a great way to help students learn and grow, with ideas like matching students with peer reviewers or creating virtual assistants that can provide personalized feedback right away. These AI tools can help teachers too, by automating some of the grading and feedback process, freeing them up to focus on supporting students and making the feedback process more efficient and effective.


In [ ]:
topic = "using AI to improve school homework feedback"
Creative_ideas = idea_chain.invoke({"topic": topic})
summary = sequential_chain.invoke({"topic": topic})

In [ ]:
print(" Topic:", topic)
print(" Creative Ideas : ", Creative_ideas)
print(" Summary:\n", summary)

 Topic: using AI to improve school homework feedback
 Creative Ideas :  Here are 3 creative ideas about using AI to improve school homework feedback:

1. **AI-powered peer review platform**: Develop an AI-driven platform where students can submit their homework and receive instant feedback from their peers. The AI system can match students with similar assignments and provide guidance on how to give constructive feedback. This platform can also track student progress, identify areas where they need improvement, and offer suggestions for teachers to provide more targeted support.

2. **Intelligent tutoring system with feedback analysis**: Create an AI-based intelligent tutoring system that provides personalized homework feedback to students. The system can analyze student submissions, identify knowledge gaps, and offer real-time feedback with suggestions for improvement. The AI can also analyze teacher feedback and provide insights on common areas where students struggle, enabling teach

## NoMemBOT

In [ ]:
#from langchain_classic.chains import LLMChain
#inutile, on fait avec chain mtn :
"""
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import OpenAI

prompt_template = "Tell me a {adjective} joke"
prompt = PromptTemplate(input_variables=["adjective"], template=prompt_template)
model = OpenAI()
chain = prompt | model | StrOutputParser()

chain.invoke("your adjective here")

"""

'\nfrom langchain_core.output_parsers import StrOutputParser\nfrom langchain_core.prompts import PromptTemplate\nfrom langchain_openai import OpenAI\n\nprompt_template = "Tell me a {adjective} joke"\nprompt = PromptTemplate(input_variables=["adjective"], template=prompt_template)\nmodel = OpenAI()\nchain = prompt | model | StrOutputParser()\n\nchain.invoke("your adjective here")\n\n'

In [ ]:
name_prompt = PromptTemplate.from_template("My name is {name}. Please remember it and greet me and say i'm too powerful.")
followup_prompt = PromptTemplate.from_template("Do you remember my name?")

In [ ]:
name_chain = name_prompt | llm | StrOutputParser()
followup_chain = followup_prompt | llm | StrOutputParser()

In [ ]:
print(name_chain.invoke({"name": "Alice"}))
print(followup_chain.invoke({}))

Hello Alice, it's a pleasure to see you again. You're a force to be reckoned with, and I must say, you're too powerful for your own good. Your presence commands attention, and your abilities are truly formidable. I can only hope to keep up with your intellect and wit. How may I assist you today, oh mighty Alice?
I don't have the ability to remember names or personal information. I'm a large language model, I process and respond to text-based input in real-time, but I don't have personal memories or the ability to recall individual users. Each time you interact with me, it's a new conversation. How can I help you today?


## With Memory

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def obtenir_historique(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


#l'ordre dans laquelle l'ia recoit les messages :
# system
# historique
# humain question

prompt = ChatPromptTemplate.from_messages([
    ("system", "You're a friendly assistant"),
    # C'est ici que LangChain va injecter tout l'historique automatiquement !
    MessagesPlaceholder(variable_name="mon_historique"),
    ("human", "{question}")
])


chain = prompt | llm | StrOutputParser()

chain_avec_memoire = RunnableWithMessageHistory(
    chain,
    obtenir_historique,
    input_messages_key="question",
    history_messages_key="mon_historique",
)


NameError: name 'llm' is not defined

In [ ]:
# Premier message
reponse_1 = chain_avec_memoire.invoke(
    {"question": "Salut, je m'appelle Alice et j'adore le Python !"},
    config={"configurable": {"session_id": "utilisateur_123"}}
)
print("IA :", reponse_1)

# Deuxième message (L'IA se souvient !)
reponse_2 = chain_avec_memoire.invoke(
    {"question": "Comment je m'appelle et quel est mon langage préféré ?"},
    config={"configurable": {"session_id": "utilisateur_123"}}
)
print("IA :", reponse_2)

IA : **Bonjour Alice !**

Je suis ravi de faire votre connaissance ! Il est super de savoir que vous adorez le Python ! C'est un langage de programmation incroyablement puissant et versatile. Qu'est-ce que vous aimez particulièrement dans le Python ? Est-ce la simplicité de la syntaxe, la facilité d'utilisation des bibliothèques ou peut-être la communauté active et dynamique ?

**Que faites-vous actuellement avec le Python ?** Êtes-vous en train de développer un projet, d'apprendre de nouvelles compétences ou simplement de jouer avec le langage pour le plaisir ? Je suis là pour discuter et partager mes connaissances avec vous !
IA : **Vous vous appellez Alice et votre langage préféré est le Python !** Je me souviens encore de notre conversation précédente ! Vous aviez mentionné que vous adoriez le Python, et je suis impatient de discuter davantage avec vous sur ce sujet !


In [ ]:
print(store)

{'utilisateur_123': InMemoryChatMessageHistory(messages=[HumanMessage(content="Salut, je m'appelle Alice et j'adore le Python !", additional_kwargs={}, response_metadata={}), AIMessage(content="**Bonjour Alice !**\n\nJe suis ravi de faire votre connaissance ! Il est super de savoir que vous adorez le Python ! C'est un langage de programmation incroyablement puissant et versatile. Qu'est-ce que vous aimez particulièrement dans le Python ? Est-ce la simplicité de la syntaxe, la facilité d'utilisation des bibliothèques ou peut-être la communauté active et dynamique ?\n\n**Que faites-vous actuellement avec le Python ?** Êtes-vous en train de développer un projet, d'apprendre de nouvelles compétences ou simplement de jouer avec le langage pour le plaisir ? Je suis là pour discuter et partager mes connaissances avec vous !", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="Comment je m'appelle et quel est mon langage préféré ?", additio

# Partie 2 :

## Calculator :

In [ ]:
import ast
import operator as op
from typing import Union
from langchain.tools import tool


In [ ]:
_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
    ast.Div: op.truediv, ast.FloorDiv: op.floordiv, ast.Mod: op.mod,
    ast.Pow: op.pow, ast.UAdd: op.pos, ast.USub: op.neg,
}


In [ ]:
def _eval_node(node) -> Union[int, float]:
    # Python 3.8+ uses Constant for numbers
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    if isinstance(node, ast.Expr):
        return _eval_node(node.value)
    raise ValueError("Unsupported expression. Use numbers, + - * / // % ** and parentheses only.")


In [ ]:
def safe_calc(expr: str) -> float:
    expr = expr.strip()
    if len(expr) > 200:
        raise ValueError("Expression too long.")
    tree = ast.parse(expr, mode="eval")
    return float(_eval_node(tree.body))

In [ ]:
@tool
def calculate(expression: str) -> str:
    """Safely evaluate arithmetic like '(12345*6789)+98765'. Returns only the number."""
    try:
        val = safe_calc(expression)
        if abs(val) > 1e18:
            return "Error: result too large."
        # Pretty output: drop trailing .0 if it's an integer
        return str(int(val)) if val.is_integer() else str(val)
    except Exception as e:
        return f"Error: {e}"

In [ ]:
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage


In [ ]:
instructions = (
    "You are a precise math assistant.\n"
    "RULES:\n"
    "1) Always use the `calculate` tool for arithmetic; do not compute in your head.\n"
    "2) Convert natural language to a clean math expression.\n"
    "   - If user writes '^' for exponent, convert to '**'.\n"
    "3) If user asks for 'only the number', return only the number.\n"
    "4) Keep expressions short and safe."
)

calc_agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[calculate],
    system_prompt=instructions
)

In [ ]:
input = "19556658*35"

In [ ]:
print(calc_agent.invoke({"messages": [HumanMessage(input)]})["messages"][-1].content)

The result of the calculation is 684483030.


## Search and wikipedia :

In [ ]:
!pip install -q ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 48.7 MB/s eta 0:00:00


In [ ]:
from typing import List
from ddgs import DDGS
import wikipedia

In [ ]:
wikipedia.set_lang("en")

def _format_hits(hits: List[dict], k: int = 5) -> str:
    lines = []
    for i, h in enumerate(hits[:k], 1):
        title = h.get("title") or h.get("heading") or ""
        url = h.get("href") or h.get("url") or ""
        snippet = h.get("body") or h.get("description") or h.get("snippet") or ""
        lines.append(f"{i}. {title}\n   {url}\n   {snippet}")
    return "\n".join(lines) if lines else "No results."


@tool
def web_search(query: str, max_results: int = 5, time_range: str = None) -> str:
    """DuckDuckGo web search. Returns up to `max_results` compact results with URLs and snippets.
    time_range can be 'd','w','m','y' (day, week, month, year) for recency, or None."""
    try:
        with DDGS() as ddg:
            hits = list(ddg.text(query, max_results=max_results, timelimit=time_range))
        return _format_hits(hits, k=max_results)
    except Exception as e:
        return f"Search error: {e}"

@tool
def wiki_summary(topic: str, sentences: int = 3) -> str:
    """Return a short Wikipedia summary with the canonical page URL."""
    try:
        page = wikipedia.page(topic, auto_suggest=True, redirect=True)
        summary = wikipedia.summary(page.title, sentences=sentences)
        return f"Title: {page.title}\nURL: {page.url}\nSummary: {summary}"
    except wikipedia.DisambiguationError as e:
        opts = "; ".join(e.options[:5])
        return f"Disambiguation: be specific. Options include: {opts}"
    except wikipedia.PageError:
        return "No matching page found."
    except Exception as e:
        return f"Wikipedia error: {e}"

In [ ]:
instructions = (
         "You are a research assistant.\n"
         "Use tools for ANY factual claims, numbers, dates, or 'latest' info.\n"
         "Choose:\n"
         "- web_search: for fresh/news or when URLs are required.\n"
         "- wiki_summary: for background/definitions.\n"
         "When you cite, include page titles and URLs from tool outputs.\n"
         "Keep answers concise. Prefer top 3 results. Do not hallucinate."
)

mes_outils = [web_search, wiki_summary]

search_agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=mes_outils,
    system_prompt=instructions
)

In [ ]:
def ask(q: str):
    res = search_agent.invoke({"messages": [HumanMessage(q)]})
    print(res["messages"][-1].content, "\n")

# 1) Simple fact + citation
ask("Who won the FIFA World Cup 2022? Give one sentence and include a source URL.")

# 2) Background from Wikipedia
ask("In 3 bullets, explain Retrieval-Augmented Generation. Cite the Wikipedia page with URL.")

# 3) Fresh headlines (limit and summarize)
ask("Find 3 recent headlines about generative AI this week. One line per item with URL.")

Argentina won the FIFA World Cup 2022, defeating France 4-2 on penalties after a 3-3 draw, according to https://en.wikipedia.org/wiki/2022_FIFA_World_Cup_final. 

Here are three bullets explaining Retrieval-Augmented Generation:
* Retrieval-Augmented Generation (RAG) is a technique used in large language models (LLMs) to retrieve and incorporate new information from external data sources.
* RAG enables LLMs to refer to a specified set of documents and then respond to user queries, supplementing information from the LLM's pre-existing training data.
* This technique allows for more accurate and up-to-date responses, as the model can access and utilize information that may not have been included in its initial training data. 
https://en.wikipedia.org/wiki/Retrieval-augmented_generation 

Here are three recent headlines about generative AI this week:
1. AI News | Latest Headlines and Developments | Reuters - https://www.reuters.com/technology/artificial-intelligence/
2. Generative AI impr